# Project Motivation

**Task** - the task is a to approximate a smooth non-linear function on a bounded 2D domain

Essentially this is a supervised regression task

As this project is for learning purposes rather pure practicality, we will be using a neural network even though there are more efficient methods of function interpolation.

### Why, can we use a neural network?
-- NEED TO ADD THAT WE CAN DO THIS USING UAT


Before we look at model architecture, training and so on, let's take a look at the data.

## Data Generation

For this project we will create synthetic data.
Here is the methodology :

We have some known smooth and non-linear function $f : [a,b]^2 \to \mathbb{R} , f(x,y)$   
where $a,b\in\mathbb{R}$

To generate the dataset, we take $n$ points $(x,y)\in[a,b]^2$
Then, we evaluate the function at these points and add some noise  
This gives us : $z_{i} = f(x_{i},y_{i}) + \epsilon_{i}\;\;1\leq i\leq n$ where $\epsilon_{i}\sim N(0,\sigma^2)\;\;i.d.d$  
This gives us the final dataset $\mathbf{D}=\{(x_{i} , y_{i} , z_{i}) \}_{i=1}^n$

This is implemented in the code below

In [ ]:
#Libaries needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
rand  = np.random.default_rng(42)

#Defining the values we talked about
a = -10
b = 10
n = int(1e5)
sigma = 2

xcoords = rand.uniform(low=a,high=b,size=n)
ycoords = rand.uniform(low=a,high=b,size=n)
labels = [] #this is the z coords produced above

#Our target function f
def f(x,y):
    return x**2 + y**2

#As the dataset is relatively small, it is fine to use a loop
for i in range(n):
    x = xcoords[i]
    y = ycoords[i]
    noise = rand.normal(loc=0 , scale=sigma)
    labels.append(f(x,y)+noise)


CoordDict = {"x" : xcoords , "y" : ycoords , "labels" : labels}
dataset = pd.DataFrame(CoordDict)

#Simple scatter plot for visualisation
fig = plt.figure()
ax = plt.axes(projection="3d")
ax.scatter(xcoords,ycoords,labels)
plt.show()

So now we have our dataset $\mathbf{D}$, we can begin to think about the model and its implementation.

We need to : 

1) Choose a model
2) Choose a loss function
3) Choose weights that makes F a close approximation of the target function

#### (1) Choosing a model
We have already decided to use a neural network but we need to think about its structure.
For now, we will consider a simple Feed-Forward Network.

##### Feed-Forward Neural Network
<img src="fnn_diagram.png" alt="Feed-Forward Neural Network Diagram"/>

I have drawn this graph to show the general structure of an FNN.

An FNN simply consists of a set of "neurons" or "nodes" (taking inspiration from neurons in the brain).

These are modelled to be basic units of computation (I will explain the exact computation later).  
  
We then group a some of these neurons together, forming what we call a "layer".  
Thinking about a network in terms of its layers
We have an input layer $\underline{x}$. Usually it is a vector containing all the features of a certain datapoint. In our example this would be a point $(x,y)$.

We then pass these values to the first layer. A layer is just a collection of neurons as shown in the diagram.  
Each neuron then computes the following proceedure:  
  
1)